# Nädal 7 — Roll C: RFM kliendisegmenteerimine

## Roll ja töö ulatus

**Roll C — RFM Analysis**

- Recency, Frequency ja Monetary arvutamine;
- R-, F- ja M-skooride määramine;
- RFM-koondskoori loomine;
- baastaseme kliendisegmentide määramine;
- segmentide kokkuvõte;
- edasijõudnute taseme kaalutud skoor, detailsemad segmendid ja CSV eksport.

### Sisend

Roll B annab puhastatud pandas DataFrame'i nimega `df`.

DataFrame peab sisaldama vähemalt järgmisi veerge:

- `customer_id`
- `sale_date`
- `sale_id`
- `total_price`

Andmete laadimine, ühendamine, puhastamine ja visualiseerimine ei kuulu selle Roll C faili koosseisu.

## 1. Teegi import ja Roll B sisend

Pandas imporditakse RFM-arvutuste tegemiseks.

> Enne järgmiste lahtrite käivitamist peab Roll A andmed imporditud ning roll B puhastatud DataFrame `df` olema notebook'is olemas.

In [ ]:
import pandas as pd

# Roll B väljund:
# df = puhastatud müügi- ja kliendiandmete DataFrame

df.head()

## 2. Baastase — RFM-mõõdikute arvutamine

### 2.1. Määra viitekuupäev: 

In [ ]:
today = pd.to_datetime("2025-02-28")

print("RFM viitekuupäev:", today)

### 2.2. Recency

Recency näitab päevade arvu kliendi viimasest ostust viitekuupäevani.

Madalam Recency väärtus on parem.

In [ ]:
recency = (
    df.groupby("customer_id")["sale_date"]
    .max()
    .reset_index()
)

recency.columns = [
    "customer_id",
    "last_purchase_date"
]

recency["recency_days"] = (
    today - recency["last_purchase_date"]
).dt.days

recency.head()

### 2.3. Frequency

Frequency näitab kliendi ostude arvu.

In [ ]:
frequency = (
    df.groupby("customer_id")["sale_id"]
    .count()
    .reset_index()
)

frequency.columns = [
    "customer_id",
    "frequency"
]

frequency.head()

### 2.4. Monetary

Monetary näitab kliendi kogukulutust.

In [ ]:
monetary = (
    df.groupby("customer_id")["total_price"]
    .sum()
    .reset_index()
)

monetary.columns = [
    "customer_id",
    "monetary_value"
]

monetary.head()

### 2.5. RFM-tabeli ühendamine

Recency, Frequency ja Monetary ühendatakse üheks kliendipõhiseks tabeliks.

In [ ]:
rfm = (
    recency[["customer_id", "recency_days"]]
    .merge(
        frequency,
        on="customer_id"
    )
    .merge(
        monetary,
        on="customer_id"
    )
)

rfm.head()

## 3. Baastase — RFM-skoorid

Iga RFM-mõõdik hinnatakse kvintiilide alusel skaalal 1–5.

- Recency: madalam väärtus saab kõrgema skoori.
- Frequency: kõrgem väärtus saab kõrgema skoori.
- Monetary: kõrgem väärtus saab kõrgema skoori.

In [ ]:
rfm["R_score"] = pd.qcut(
    rfm["recency_days"],
    5,
    labels=[5, 4, 3, 2, 1]
)

rfm["F_score"] = pd.qcut(
    rfm["frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm["M_score"] = pd.qcut(
    rfm["monetary_value"],
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm["R_score"] = rfm["R_score"].astype(int)
rfm["F_score"] = rfm["F_score"].astype(int)
rfm["M_score"] = rfm["M_score"].astype(int)

rfm["RFM_Score"] = (
    rfm["R_score"]
    + rfm["F_score"]
    + rfm["M_score"]
)

rfm.head()

## 4. Baastase — viis kliendisegmenti

Juhendi baastaseme segmendid:

| RFM-skoor | Segment |
|---:|---|
| 13–15 | VIP Champions |
| 10–12 | Loyal |
| 7–9 | Potential |
| 4–6 | At Risk |
| 3 | Lost |

In [ ]:
def segment_customer(row):
    if row["RFM_Score"] >= 13:
        return "VIP Champions"
    elif row["RFM_Score"] >= 10:
        return "Loyal"
    elif row["RFM_Score"] >= 7:
        return "Potential"
    elif row["RFM_Score"] >= 4:
        return "At Risk"
    else:
        return "Lost"


rfm["Segment"] = rfm.apply(
    segment_customer,
    axis=1
)

rfm.head()

## 5. Baastaseme kokkuvõte

Kokkuvõte näitab iga segmendi klientide arvu ja osakaalu.

In [ ]:
segment_summary = (
    rfm["Segment"]
    .value_counts()
    .rename_axis("Segment")
    .reset_index(name="customers")
)

segment_summary["customer_share_pct"] = (
    segment_summary["customers"]
    / len(rfm)
    * 100
)

segment_summary["customer_share_pct"] = (
    segment_summary["customer_share_pct"]
    .round(2)
)

segment_summary

### Baastaseme kvaliteedikontroll

Kontrollitakse juhendis nõutud tingimusi:

- RFM-väärtused on arvutatud;
- skoorid jäävad vahemikku 1–5;
- iga klient sai segmendi;
- kokkuvõte näitab klientide arvu ja osakaalu.

In [ ]:
print("Kliente RFM-tabelis:", len(rfm))
print("Segmendita kliente:", rfm["Segment"].isna().sum())

print("\nSkooride vahemikud:")
print(
    rfm[
        ["R_score", "F_score", "M_score"]
    ].agg(["min", "max"])
)

print(
    "\nKlientide osakaal kokku:",
    round(segment_summary["customer_share_pct"].sum(), 2),
    "%"
)

# Edasijõudnute tase

Juhendi vabatahtlik edasijõudnute osa sisaldab:

1. Monetary kahekordse kaaluga skoori;
2. kuut detailsemat kliendisegmenti;
3. segmentide eksporti CSV-failina.

## 6. Kaalutud RFM-skoor

Monetary saab kahekordse kaalu, sest kliendi kulutus on Marko jaoks olulisem.

Kaalutud skoor:

`R_score + F_score + 2 × M_score`

In [ ]:
rfm["Weighted_RFM_Score"] = (
    rfm["R_score"]
    + rfm["F_score"]
    + 2 * rfm["M_score"]
)

rfm[
    [
        "customer_id",
        "R_score",
        "F_score",
        "M_score",
        "RFM_Score",
        "Weighted_RFM_Score"
    ]
].head()

## 7. Detailsemad segmendid

Juhendi edasijõudnute segmendid:

| RFM-skoor | Segment | Tegevus |
|---:|---|---|
| 13–15 | VIP Champions | Early access, VIP sooduskoodid |
| 11–12 | Loyal Customers | Lojaalsusprogramm, preemiad |
| 9–10 | Regular Customers | Cross-sell kampaaniad |
| 7–8 | New Customers | Onboarding, welcome-sari |
| 5–6 | At Risk | Win-back kampaania, personaliseeritud e-mail |
| 3–4 | Lost | Viimane katse, suur soodustus |

In [ ]:
def assign_advanced_segment(row):
    if row["RFM_Score"] >= 13:
        return "VIP Champions"
    elif row["RFM_Score"] >= 11:
        return "Loyal Customers"
    elif row["RFM_Score"] >= 9:
        return "Regular Customers"
    elif row["RFM_Score"] >= 7:
        return "New Customers"
    elif row["RFM_Score"] >= 5:
        return "At Risk"
    else:
        return "Lost"


rfm["Advanced_Segment"] = rfm.apply(
    assign_advanced_segment,
    axis=1
)

rfm.head()

## 8. Edasijõudnute segmentide kokkuvõte

In [ ]:
advanced_segment_summary = (
    rfm["Advanced_Segment"]
    .value_counts()
    .rename_axis("Advanced_Segment")
    .reset_index(name="customers")
)

advanced_segment_summary["customer_share_pct"] = (
    advanced_segment_summary["customers"]
    / len(rfm)
    * 100
)

advanced_segment_summary["customer_share_pct"] = (
    advanced_segment_summary["customer_share_pct"]
    .round(2)
)

advanced_segment_summary

## 9. Tulemuste eksport

Juhendi järgi salvestatakse segmendid faili `rfm_segments.csv`, et Marko saaks tulemuse turundusmeeskonnale edastada.

In [ ]:
rfm.to_csv(
    "rfm_segments.csv",
    index=False
)

print("Fail salvestatud: rfm_segments.csv")

## 10. Roll C väljund

Roll C annab Roll D-le edasi DataFrame'i `rfm`, mis sisaldab:

- `customer_id`;
- `recency_days`;
- `frequency`;
- `monetary_value`;
- R-, F- ja M-skoore;
- baastaseme RFM-koondskoori ja segmenti;
- edasijõudnute kaalutud skoori ja detailsemat segmenti.

Roll D kasutab seda tabelit visualiseerimiseks ja äritõlgenduse koostamiseks.